In [ ]:
import os, gc, csv, random, pickle, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from scipy.special import softmax
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    f1_score, matthews_corrcoef, classification_report,
    accuracy_score, precision_recall_fscore_support,
    roc_auc_score, roc_curve, confusion_matrix, ConfusionMatrixDisplay
)
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification, Trainer,
    TrainingArguments, DataCollatorWithPadding, EarlyStoppingCallback
)
from datasets import Dataset

# ==========================================
# 1. PATHS & RESEARCH CONFIG
# ==========================================
BASE_OUT_DIR = "*insert path here*"
TRAIN_CSV = "*insert path here*"
RAND_CSV  = "*insert path here*"
TEST_CSV  = "*insert path here*"

EXPERIMENTAL_SEEDS = [2026, 2027, 2028, 2029, 2030] # 5-seed per run, change manually each run when finished e.g. [2031, 2032, 2033, 2034, 2035], [2036, 2037, 2038, 2039, 2040], ...
MAX_LEN    = 128
LR_DEFAULT = 2e-5

BASE_MODELS = [
    {"alias": "indobert-p1",   "path": "indobenchmark/indobert-base-p1"},
    {"alias": "indobertweet",  "path": "indolem/indobertweet-base-uncased"},
    {"alias": "mdeberta-v3",   "path": "microsoft/mdeberta-v3-base"},
]


In [ ]:
# ==========================================
# 2. UTILITIES
# ==========================================
def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

def clean_text(text):
    if not isinstance(text, str): return ""
    return text.replace("\n", " ").strip()

def preprocess_function(examples, tokenizer):
    return tokenizer(examples["text"], truncation=True, padding=False, max_length=MAX_LEN)

def compute_metrics(eval_pred):
    """Full metrics matching reference notebook: accuracy, precision, recall, F1, MCC, AUC."""
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    prec, rec, f1, _ = precision_recall_fscore_support(labels, preds, average='binary', zero_division=0)
    acc = accuracy_score(labels, preds)
    mcc = matthews_corrcoef(labels, preds)
    probs = softmax(logits, axis=-1)[:, 1]
    try:
        auc = roc_auc_score(labels, probs)
    except ValueError:
        auc = float('nan')
    return {
        "accuracy":  float(acc),
        "precision": float(prec),
        "recall":    float(rec),
        "f1":        float(f1),
        "mcc":       float(mcc),
        "auc":       float(auc),
    }

def compute_ece(probs, labels, n_bins=10):
    """
    Expected Calibration Error.
    probs  : predicted P(positive) as a 1-D array
    labels : true binary labels
    """
    bins = np.linspace(0, 1, n_bins + 1)
    ece  = 0.0
    for i in range(n_bins):
        lo, hi = bins[i], bins[i + 1]
        mask = (probs >= lo) & (probs < hi)
        if mask.sum() == 0:
            continue
        bin_acc  = labels[mask].mean()
        bin_conf = probs[mask].mean()
        ece += (mask.sum() / len(probs)) * abs(bin_acc - bin_conf)
    return float(ece)

def plot_reliability(probs, labels, title, n_bins=10, save_path=None):
    """Reliability (calibration) diagram."""
    bins      = np.linspace(0, 1, n_bins + 1)
    bin_confs, bin_accs, bin_sizes = [], [], []
    for i in range(n_bins):
        lo, hi = bins[i], bins[i + 1]
        mask = (probs >= lo) & (probs < hi)
        bin_sizes.append(mask.sum())
        if mask.sum() == 0:
            bin_confs.append(None); bin_accs.append(None)
        else:
            bin_confs.append(float(probs[mask].mean()))
            bin_accs.append(float(labels[mask].mean()))
    ece = compute_ece(probs, labels, n_bins)
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Perfect calibration')
    valid = [(c, a) for c, a in zip(bin_confs, bin_accs) if c is not None]
    if valid:
        cs, as_ = zip(*valid)
        ax.plot(cs, as_, marker='o', color='steelblue', lw=2, label='Model')
    ax.set_xlabel('Mean Predicted Probability (Confidence)')
    ax.set_ylabel('Fraction of Positives (Accuracy)')
    ax.set_title(f'{title}\nECE = {ece:.4f}')
    ax.legend(); ax.grid(alpha=0.3)
    fig.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=150)
    plt.show()
    return ece


In [ ]:
# ==========================================
# 3. EXECUTION ENGINE
# ==========================================
all_seeds_summary = []

for SEED in EXPERIMENTAL_SEEDS:
    print(f"\n{'='*60}\nRUNNING EXPERIMENT SEED: {SEED}\n{'='*60}")
    set_seed(SEED)

    CURRENT_SEED_DIR = os.path.join(BASE_OUT_DIR, f"run_seed_{SEED}")
    RESULTS_DIR = os.path.join(CURRENT_SEED_DIR, "results")
    os.makedirs(RESULTS_DIR, exist_ok=True)

    # --- 1. Load & Clean Data ---
    df1 = pd.read_csv(TRAIN_CSV)
    df2 = pd.read_csv(RAND_CSV)
    full_train = pd.concat([df1, df2], ignore_index=True)
    df_test = pd.read_csv(TEST_CSV)

    initial_len = len(full_train)
    full_train['label'] = pd.to_numeric(full_train['label'], errors='coerce')
    full_train = full_train.dropna(subset=['label', 'text'])
    full_train['label'] = full_train['label'].astype(int)
    full_train = full_train.reset_index(drop=True)

    df_test['label'] = pd.to_numeric(df_test['label'], errors='coerce')
    df_test = df_test.dropna(subset=['label', 'text'])
    df_test['label'] = df_test['label'].astype(int)
    df_test = df_test.reset_index(drop=True)

    if len(full_train) < initial_len:
        print(f"!!! Cleaned {initial_len - len(full_train)} invalid rows from training data.")

    # --- 2. Per-model cross-validated OOF + test probabilities ---
    oof_probs      = np.zeros((len(full_train), len(BASE_MODELS)))
    test_probs_agg = np.zeros((len(df_test),    len(BASE_MODELS)))
    # Stores fold-level metrics per model, used in box plot later
    per_model_fold_metrics = {cfg['alias']: [] for cfg in BASE_MODELS}
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

    for m_idx, cfg in enumerate(BASE_MODELS):
        print(f"\n>> Model: {cfg['alias']}")
        tokenizer = AutoTokenizer.from_pretrained(cfg['path'])
        test_ds = Dataset.from_pandas(df_test).map(
            lambda x: preprocess_function(x, tokenizer), batched=True
        )
        fold_test_preds = []

        for fold, (t_idx, v_idx) in enumerate(skf.split(full_train['text'], full_train['label'])):
            print(f"   Fold {fold+1}/5")
            train_ds = Dataset.from_pandas(full_train.iloc[t_idx]).map(
                lambda x: preprocess_function(x, tokenizer), batched=True
            )
            val_ds = Dataset.from_pandas(full_train.iloc[v_idx]).map(
                lambda x: preprocess_function(x, tokenizer), batched=True
            )

            model = AutoModelForSequenceClassification.from_pretrained(cfg['path'], num_labels=2)
            args = TrainingArguments(
                output_dir="tmp_checkpoints",
                per_device_train_batch_size=16,
                num_train_epochs=11,
                learning_rate=LR_DEFAULT,
                eval_strategy="steps",
                eval_steps=50,
                save_strategy="steps",
                save_steps=50,
                save_total_limit=1,
                load_best_model_at_end=True,
                metric_for_best_model="f1",
                report_to="none",
                log_level="error",
                disable_tqdm=True
            )
            trainer = Trainer(
                model=model,
                args=args,
                train_dataset=train_ds,
                eval_dataset=val_ds,
                data_collator=DataCollatorWithPadding(tokenizer),
                compute_metrics=compute_metrics,
                callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
            )
            trainer.train()

            # Collect OOF predictions
            oof_output = trainer.predict(val_ds)
            oof_probs[v_idx, m_idx] = softmax(oof_output.predictions, axis=-1)[:, 1]

            # Store fold-level metrics for box plot
            fold_m = compute_metrics((oof_output.predictions, full_train['label'].values[v_idx]))
            fold_m['fold'] = fold + 1
            per_model_fold_metrics[cfg['alias']].append(fold_m)

            # Collect test predictions
            test_output = trainer.predict(test_ds)
            fold_test_preds.append(softmax(test_output.predictions, axis=-1)[:, 1])

            del model, trainer
            gc.collect()
            torch.cuda.empty_cache()

        # Average test predictions across folds for this model
        test_probs_agg[:, m_idx] = np.mean(fold_test_preds, axis=0)

    # --- 3. Save raw OOF and test probabilities ---
    pd.DataFrame(oof_probs, columns=[m['alias'] for m in BASE_MODELS]).to_csv(
        os.path.join(RESULTS_DIR, f"oof_train_{SEED}.csv"), index=False
    )
    pd.DataFrame(test_probs_agg, columns=[m['alias'] for m in BASE_MODELS]).to_csv(
        os.path.join(RESULTS_DIR, f"probs_test_{SEED}.csv"), index=False
    )

    # --- 4. Meta-learner: find best threshold on OOF, apply once to test ---
    # FIX: lr_meta.fit() and threshold search happen AFTER the loop fills oof_probs
    lr_meta = LogisticRegression(max_iter=1000)
    lr_meta.fit(oof_probs, full_train['label'].values)

    oof_meta_probs = lr_meta.predict_proba(oof_probs)[:, 1]
    thresholds = np.arange(0.3, 0.99, 0.01)
    best_thresh = max(
        thresholds,
        key=lambda t: f1_score(full_train['label'], (oof_meta_probs >= t).astype(int))
    )
    print(f"Best OOF threshold for seed {SEED}: {best_thresh:.2f}")

    final_probs  = lr_meta.predict_proba(test_probs_agg)[:, 1]
    final_labels = (final_probs >= best_thresh).astype(int)
    true_labels  = df_test['label'].values

    # --- 5. Full ensemble metrics ---
    f1   = f1_score(true_labels, final_labels)
    mcc  = matthews_corrcoef(true_labels, final_labels)
    prec, rec, _, _ = precision_recall_fscore_support(true_labels, final_labels, average='binary', zero_division=0)
    acc  = accuracy_score(true_labels, final_labels)
    try:
        auc = roc_auc_score(true_labels, final_probs)
    except ValueError:
        auc = float('nan')
    print(f"Seed {SEED} — F1: {f1:.4f} | MCC: {mcc:.4f} | Acc: {acc:.4f} | AUC: {auc:.4f}")

    # --- 6. Per-model test metrics (at 0.5 threshold) ---
    per_model_test_metrics = {}
    for m_idx, cfg in enumerate(BASE_MODELS):
        col_probs  = test_probs_agg[:, m_idx]
        col_labels = (col_probs >= 0.5).astype(int)
        p, r, f, _ = precision_recall_fscore_support(true_labels, col_labels, average='binary', zero_division=0)
        try:
            m_auc = roc_auc_score(true_labels, col_probs)
        except ValueError:
            m_auc = float('nan')
        per_model_test_metrics[cfg['alias']] = {
            "f1":        float(f),
            "mcc":       float(matthews_corrcoef(true_labels, col_labels)),
            "accuracy":  float(accuracy_score(true_labels, col_labels)),
            "precision": float(p),
            "recall":    float(r),
            "auc":       float(m_auc),
        }

    # --- 7. ECE for ensemble and each base model ---
    ensemble_ece = compute_ece(final_probs, true_labels)
    per_model_ece = {}
    for m_idx, cfg in enumerate(BASE_MODELS):
        col_probs = test_probs_agg[:, m_idx]
        per_model_ece[cfg['alias']] = compute_ece(col_probs, true_labels)
    print(f"Ensemble ECE: {ensemble_ece:.4f}")
    for a, v in per_model_ece.items(): print(f"  {a} ECE: {v:.4f}")

    all_seeds_summary.append({
        "Seed": SEED, "F1": f1, "MCC": mcc,
        "Accuracy": acc, "Precision": prec, "Recall": rec, "AUC": auc,
        "Threshold": best_thresh,
        "per_model": per_model_test_metrics,
        "per_model_fold_metrics": per_model_fold_metrics,
        "true_labels":    true_labels.tolist(),
        "final_probs":    final_probs.tolist(),
        "final_labels":   final_labels.tolist(),
        "oof_meta_probs": oof_meta_probs.tolist(),
        "oof_true_labels": full_train['label'].values.tolist(),
        "ensemble_ece":  ensemble_ece,
        "per_model_ece": per_model_ece,
    })

    # Append scalar fields only to running CSV
    scalar_keys = {'Seed','F1','MCC','Accuracy','Precision','Recall','AUC','Threshold'}
    row_df = pd.DataFrame([{k: v for k, v in all_seeds_summary[-1].items() if k in scalar_keys}])
    progress_csv = os.path.join(BASE_OUT_DIR, "overall_experiment_progress.csv")
    row_df.to_csv(progress_csv, mode='a', header=not os.path.exists(progress_csv), index=False)


## Results & Analysis
Cells below read from `all_seeds_summary` populated during training.
They can also be re-run independently after loading results from disk.

In [ ]:
# ==========================================
# 4. FINAL SUMMARY TABLE
# ==========================================
summary_df = pd.read_csv(os.path.join(BASE_OUT_DIR, 'overall_experiment_progress.csv'))
print(f"\n{'='*40}\nFINAL EXPERIMENTAL SUMMARY\n{'='*40}")
print(summary_df[['Seed','F1','MCC','Accuracy','Precision','Recall','AUC','Threshold']].to_string(index=False))
print(f"\nMean F1 : {summary_df['F1'].mean():.4f}  (+/- {summary_df['F1'].std():.4f})")
print(f"Mean MCC: {summary_df['MCC'].mean():.4f}  (+/- {summary_df['MCC'].std():.4f})")
print(f"Mean AUC: {summary_df['AUC'].mean():.4f}  (+/- {summary_df['AUC'].std():.4f})")

# ECE summary from in-memory results
print(f"\n{'='*40}\nECE SUMMARY (Ensemble)\n{'='*40}")
ece_vals = [e['ensemble_ece'] for e in all_seeds_summary]
for entry in all_seeds_summary:
    print(f"  Seed {entry['Seed']}: ECE = {entry['ensemble_ece']:.4f}")
print(f"Mean ECE: {np.mean(ece_vals):.4f}  (+/- {np.std(ece_vals):.4f})")
summary_df


In [ ]:
# ==========================================
# 5. PER-MODEL vs ENSEMBLE BAR CHART (F1 & MCC)
# ==========================================
model_aliases = [m['alias'] for m in BASE_MODELS]
agg = {a: {k: [] for k in ['f1','mcc','accuracy','precision','recall','auc','ece']} for a in model_aliases}
ens = {k: [] for k in ['f1','mcc','accuracy','precision','recall','auc','ece']}

for entry in all_seeds_summary:
    ens['f1'].append(entry['F1']);        ens['mcc'].append(entry['MCC'])
    ens['accuracy'].append(entry['Accuracy'])
    ens['precision'].append(entry['Precision']); ens['recall'].append(entry['Recall'])
    ens['auc'].append(entry['AUC'])
    ens['ece'].append(entry['ensemble_ece'])
    for a in model_aliases:
        for k in ['f1','mcc','accuracy','precision','recall','auc']:
            agg[a][k].append(entry['per_model'][a][k])
        agg[a]['ece'].append(entry['per_model_ece'][a])

rows = []
for a in model_aliases:
    rows.append({"label": a, "F1": np.mean(agg[a]['f1']), "MCC": np.mean(agg[a]['mcc']),
                 "accuracy": np.mean(agg[a]['accuracy']), "precision": np.mean(agg[a]['precision']),
                 "recall": np.mean(agg[a]['recall']), "auc": np.mean(agg[a]['auc']),
                 "ece": np.mean(agg[a]['ece'])})
rows.append({"label": "STACK (Ensemble)", "F1": np.mean(ens['f1']), "MCC": np.mean(ens['mcc']),
             "accuracy": np.mean(ens['accuracy']), "precision": np.mean(ens['precision']),
             "recall": np.mean(ens['recall']), "auc": np.mean(ens['auc']),
             "ece": np.mean(ens['ece'])})
rows = sorted(rows, key=lambda r: r['F1'], reverse=True)
metrics_df = pd.DataFrame(rows)

labels_plot = [r['label']  for r in rows]
f1_vals     = [r['F1']     for r in rows]
mcc_vals    = [r['MCC']    for r in rows]
x = np.arange(len(labels_plot)); w = 0.38

fig5, ax = plt.subplots(figsize=(11, 4.5))
ax.bar(x - w/2, f1_vals,  width=w, label='F1',  color='steelblue')
ax.bar(x + w/2, mcc_vals, width=w, label='MCC', color='coral')
ax.set_xticks(x)
ax.set_xticklabels(labels_plot, rotation=25, ha='right')
ax.set_ylabel('Score (TEST — averaged across seeds)')
ax.set_ylim(0, 1.0)
ax.legend(loc='upper left')
ax.grid(axis='y', linestyle='--', alpha=0.4)
ax.set_title('Per-Model vs Ensemble: F1 & MCC')
for xi, v in zip(x - w/2, f1_vals):  ax.text(xi, v + 0.012, f'{v:.3f}', ha='center', va='bottom', fontsize=8)
for xi, v in zip(x + w/2, mcc_vals): ax.text(xi, v + 0.012, f'{v:.3f}', ha='center', va='bottom', fontsize=8)
fig5.tight_layout()
fig5.savefig(os.path.join(BASE_OUT_DIR, 'fig5_base_vs_ensemble.png'), dpi=150)
plt.show()
print(metrics_df.to_string(index=False))


In [ ]:
# ==========================================
# 6. ROC CURVES — ENSEMBLE ACROSS SEEDS
# ==========================================
fig6, ax = plt.subplots(figsize=(7, 6))
colors   = plt.cm.tab10.colors
mean_fpr = np.linspace(0, 1, 200)
tprs     = []

for i, entry in enumerate(all_seeds_summary):
    tl = np.array(entry['true_labels'])
    fp = np.array(entry['final_probs'])
    if len(np.unique(tl)) < 2: continue
    fpr, tpr, _ = roc_curve(tl, fp)
    auc_val = roc_auc_score(tl, fp)
    interp_tpr = np.interp(mean_fpr, fpr, tpr); interp_tpr[0] = 0.0
    tprs.append(interp_tpr)
    ax.plot(fpr, tpr, alpha=0.5, lw=1.2, color=colors[i % 10],
            label=f"Seed {entry['Seed']} (AUC={auc_val:.3f})")

if tprs:
    mean_tpr = np.mean(tprs, axis=0); mean_tpr[-1] = 1.0
    mean_auc = np.mean([roc_auc_score(np.array(e['true_labels']), np.array(e['final_probs']))
                        for e in all_seeds_summary])
    ax.plot(mean_fpr, mean_tpr, color='black', lw=2.5, linestyle='--',
            label=f'Mean ROC (AUC={mean_auc:.3f})')

ax.plot([0,1],[0,1],'k:',lw=1)
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve — Stacked Ensemble (All Seeds)')
ax.legend(loc='lower right', fontsize=8); ax.grid(alpha=0.3)
fig6.tight_layout()
fig6.savefig(os.path.join(BASE_OUT_DIR, 'fig6_roc_curves.png'), dpi=150)
plt.show()


In [ ]:
# ==========================================
# 7. CONFUSION MATRICES — PER SEED
# ==========================================
n = len(all_seeds_summary)
ncols = min(n, 3); nrows = (n + ncols - 1) // ncols
fig7, axes = plt.subplots(nrows, ncols, figsize=(5*ncols, 4.5*nrows))
axes = np.array(axes).flatten()

for i, entry in enumerate(all_seeds_summary):
    tl = np.array(entry['true_labels'])
    fl = np.array(entry['final_labels'])
    cm = confusion_matrix(tl, fl)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Not-Depressed','Depressed'])
    disp.plot(ax=axes[i], colorbar=False, cmap='Blues')
    axes[i].set_title(f"Seed {entry['Seed']}  |  F1={entry['F1']:.3f}  MCC={entry['MCC']:.3f}", fontsize=9)

for j in range(i+1, len(axes)): axes[j].set_visible(False)
fig7.suptitle('Confusion Matrices — Stacked Ensemble', fontsize=13, y=1.01)
fig7.tight_layout()
fig7.savefig(os.path.join(BASE_OUT_DIR, 'fig7_confusion_matrices.png'), dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ==========================================
# 8. PER-MODEL OOF F1 DISTRIBUTION — BOX PLOT
# ==========================================
fold_f1_by_model = {a: [] for a in model_aliases}
for entry in all_seeds_summary:
    for a in model_aliases:
        fold_f1_by_model[a].extend([fm['f1'] for fm in entry['per_model_fold_metrics'][a]])

fig8, ax = plt.subplots(figsize=(10, 5))
bp = ax.boxplot(
    [fold_f1_by_model[a] for a in model_aliases],
    labels=model_aliases, patch_artist=True,
    medianprops=dict(color='black', linewidth=2)
)
for patch, color in zip(bp['boxes'], plt.cm.tab10.colors):
    patch.set_facecolor(color); patch.set_alpha(0.7)
ax.set_ylabel('OOF F1 Score (per fold × seed)')
ax.set_title('Per-Model OOF F1 Distribution Across Folds & Seeds')
ax.grid(axis='y', linestyle='--', alpha=0.4)
plt.xticks(rotation=20, ha='right')
fig8.tight_layout()
fig8.savefig(os.path.join(BASE_OUT_DIR, 'fig8_fold_f1_boxplot.png'), dpi=150)
plt.show()


In [ ]:
# ==========================================
# 9. SEED STABILITY — F1 & MCC LINE PLOT
# ==========================================
summary_df = pd.read_csv(os.path.join(BASE_OUT_DIR, 'overall_experiment_progress.csv'))

fig9, ax = plt.subplots(figsize=(8, 4))
ax.plot(summary_df['Seed'], summary_df['F1'],  marker='o', label='F1',  color='steelblue', lw=2)
ax.plot(summary_df['Seed'], summary_df['MCC'], marker='s', label='MCC', color='coral',     lw=2)
ax.axhline(summary_df['F1'].mean(),  linestyle='--', color='steelblue', alpha=0.5,
           label=f"Mean F1={summary_df['F1'].mean():.4f}")
ax.axhline(summary_df['MCC'].mean(), linestyle='--', color='coral',     alpha=0.5,
           label=f"Mean MCC={summary_df['MCC'].mean():.4f}")
ax.set_xlabel('Seed'); ax.set_ylabel('Score')
ax.set_title('Ensemble Stability Across Seeds')
ax.legend(fontsize=9); ax.grid(alpha=0.3)
fig9.tight_layout()
fig9.savefig(os.path.join(BASE_OUT_DIR, 'fig9_seed_stability.png'), dpi=150)
plt.show()


In [ ]:
# ==========================================
# 11. ECE ANALYSIS
# ==========================================

# --- 11a. ECE table: ensemble vs per-model, averaged across seeds ---
model_aliases = [m['alias'] for m in BASE_MODELS]
ece_rows = []
for a in model_aliases:
    ece_rows.append({
        'label': a,
        'mean_ece': np.mean([e['per_model_ece'][a] for e in all_seeds_summary]),
        'std_ece':  np.std( [e['per_model_ece'][a] for e in all_seeds_summary]),
    })
ece_rows.append({
    'label':    'STACK (Ensemble)',
    'mean_ece': np.mean([e['ensemble_ece'] for e in all_seeds_summary]),
    'std_ece':  np.std( [e['ensemble_ece'] for e in all_seeds_summary]),
})
ece_df = pd.DataFrame(ece_rows).sort_values('mean_ece')
print('ECE Summary (lower = better calibrated):')
print(ece_df.to_string(index=False))

# --- 11b. ECE bar chart: per-model + ensemble ---
fig11a, ax = plt.subplots(figsize=(10, 4.5))
labels_ece = ece_df['label'].tolist()
means_ece  = ece_df['mean_ece'].tolist()
stds_ece   = ece_df['std_ece'].tolist()
colors_ece = ['steelblue' if l != 'STACK (Ensemble)' else 'darkorange' for l in labels_ece]
xp = np.arange(len(labels_ece))
bars = ax.bar(xp, means_ece, yerr=stds_ece, capsize=4,
              color=colors_ece, alpha=0.85, width=0.55)
ax.set_xticks(xp)
ax.set_xticklabels(labels_ece, rotation=25, ha='right')
ax.set_ylabel('ECE (lower = better calibrated)')
ax.set_title('Expected Calibration Error — Per Model vs Ensemble')
ax.grid(axis='y', linestyle='--', alpha=0.4)
for xi, v in zip(xp, means_ece):
    ax.text(xi, v + 0.001, f'{v:.4f}', ha='center', va='bottom', fontsize=8)
fig11a.tight_layout()
fig11a.savefig(os.path.join(BASE_OUT_DIR, 'fig11a_ece_bar.png'), dpi=150)
plt.show()

# --- 11c. Reliability diagrams — ensemble, one per seed ---
for entry in all_seeds_summary:
    plot_reliability(
        np.array(entry['final_probs']),
        np.array(entry['true_labels']),
        title=f'Reliability Diagram — Ensemble  (Seed { entry["Seed"]})',
        save_path=os.path.join(BASE_OUT_DIR, f'fig11b_reliability_ensemble_seed{entry["Seed"]}.png')
    )

# --- 11d. Reliability diagrams — per model (averaged seed probabilities) ---
print('\nPer-model reliability diagrams (probabilities averaged across seeds)...')
for m_idx, cfg in enumerate(BASE_MODELS):
    # Average test probs for this model across all seeds from saved CSVs
    all_probs = []
    true_lbl  = np.array(all_seeds_summary[0]['true_labels'])  # same test set across seeds
    for entry in all_seeds_summary:
        probs_path = os.path.join(
            BASE_OUT_DIR, f"run_seed_{entry['Seed']}", "results", f"probs_test_{entry['Seed']}.csv"
        )
        df_probs = pd.read_csv(probs_path)
        all_probs.append(df_probs[cfg['alias']].values)
    mean_probs = np.mean(all_probs, axis=0)
    plot_reliability(
        mean_probs, true_lbl,
        title=f'Reliability Diagram — {cfg["alias"]} (mean across seeds)',
        save_path=os.path.join(BASE_OUT_DIR, f'fig11c_reliability_{cfg["alias"]}.png')
    )

In [ ]:
# ==========================================
# 12. EXPORT FULL METRICS TABLE (includes ECE)
# ==========================================
out_path = os.path.join(BASE_OUT_DIR, 'metrics_comparison_test.csv')
metrics_df.to_csv(out_path, index=False)
print('Saved metrics comparison table to:', out_path)

ece_path = os.path.join(BASE_OUT_DIR, 'ece_comparison.csv')
ece_df.to_csv(ece_path, index=False)
print('Saved ECE comparison table to:', ece_path)

metrics_df
